# DNABERT-2 test

To make it work, install all the packages in the `requirements-dnabert.txt` file and run **pip uninstall triton**. If this library is installed, the code will fail. Another way to fix it is to download the `DNABERT-2-117M` repo and in the file `flash_attn_triton.py` change all the *tl.dot(q, k, trans_b=True)* with *tl.dot(q, tl.trans(k))*, and install the latest version available of triton (even if pytorch complains). Please note that with triton installed it will only work on the GPU.

In [1]:
import pandas as pd
import os

## Install dependencies

In [2]:
!git clone https://huggingface.co/zhihan1996/DNABERT-2-117M

Cloning into 'DNABERT-2-117M'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 71 (delta 0), reused 0 (delta 0), pack-reused 68 (from 1)
Unpacking objects: 100% (71/71), 74.78 KiB | 2.58 MiB/s, done.


Now, in the file `flash_attn_triton.py` change all the *tl.dot(q, k, trans_b=True)* with *tl.dot(q, tl.trans(k))*.

In [3]:
%pip install -r requirements-dnabert.txt
%pip uninstall triton -y
%pip install triton

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 87.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.2/475.2 MB 97.7 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 670.2/670.2 MB 88.0 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 96.8 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 98.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 96.7 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 77.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 85.0 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 104.0 MB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 100.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 102.9 MB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.

## Load data

In [8]:
base_path = "../../data/perphect-data"

couples_df = pd.read_csv(os.path.join(base_path, "public_data_set", "couples_df.csv"))
phages_df = pd.read_csv(os.path.join(base_path, "public_data_set", "phages_df.csv"))
bacteria_df = pd.read_csv(os.path.join(base_path, "public_data_set", "bacteria_df.csv"))
print('couples_df.shape = ', couples_df.shape)
print('phages_df.shape = ', phages_df.shape)
print('bacteria_df.shape = ', bacteria_df.shape)

couples_df.shape =  (4202, 4)
phages_df.shape =  (3459, 3)
bacteria_df.shape =  (94, 3)


In [ ]:
phages_df.head()

## Get embedding

In [2]:
import torch
torch.cuda.is_available()

True

In [3]:
import gc
from transformers.models.bert.configuration_bert import BertConfig
from transformers import AutoTokenizer, AutoModel
import torch.nn as nn

In [4]:
device = "cuda:3" if torch.cuda.is_available() else "cpu"
def clean_gpu():
    torch.cuda.empty_cache()
    gc.collect()
clean_gpu()
device

'cuda:3'

In [5]:
clean_gpu()
tokenizer = AutoTokenizer.from_pretrained("DNABERT-2-117M/", trust_remote_code=True)
config = BertConfig.from_pretrained("DNABERT-2-117M/")
model = AutoModel.from_pretrained("DNABERT-2-117M/", trust_remote_code=True, config=config)

/home/pere.carrillo/micromamba/envs/pbi/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
Some weights of BertModel were not initialized from the model checkpoint at DNABERT-2-117M/ and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
model.to(device)

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(4096, 768, padding_idx=0)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertUnpadAttention(
          (self): BertUnpadSelfAttention(
            (dropout): Dropout(p=0.0, inplace=False)
            (Wqkv): Linear(in_features=768, out_features=2304, bias=True)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
        (mlp): BertGatedLinearUnitMLP(
          (gated_layers): Linear(in_features=768, out_features=6144, bias=False)
          (act): GELU(approximate='none')
  

In [9]:
clean_gpu()
# dna = "ACGTAGCATCGGATCTATCTATCGACACTTGGTTATCGATCTACGAGCATCTCGTTAGC"
# dna = phages_df["phage_sequence"].iloc[0]
dna = bacteria_df["bacterium_sequence"].iloc[0][:2**15] # limit to 32k bps as it is the maximum that fits in a A40 GPU
inputs = tokenizer(dna, return_tensors = 'pt')["input_ids"].to(device)
hidden_states = model(inputs)[0] # [1, sequence_length, 768]

# embedding with mean pooling
embedding_mean = torch.mean(hidden_states[0], dim=0)
print(embedding_mean.shape) # expect to be 768

# embedding with max pooling
embedding_max = torch.max(hidden_states[0], dim=0)[0]
print(embedding_max.shape) # expect to be 768

/home/pere.carrillo/.cache/huggingface/modules/transformers_modules/bert_layers.py:433: UserWarning: Increasing alibi size from 512 to 7418
  warnings.warn(


torch.Size([768])
torch.Size([768])


In [ ]:
dna = [bacteria_df["bacterium_sequence"].iloc[0][:2**15], bacteria_df["bacterium_sequence"].iloc[1][:2**15]] # limit to 32k bps as it is the maximum that fits in a A40 GPU
inputs = tokenizer(dna, return_tensors = 'pt', padding=True)["input_ids"].to(device)
inputs.shape

torch.Size([2, 14839])

In [19]:
with torch.no_grad():
    hidden_states = model(inputs[0].unsqueeze(0))[0] # [1, sequence_length, 768]

# embedding with mean pooling
embedding_mean = torch.mean(hidden_states[0], dim=0)
print(embedding_mean.shape) # expect to be 768

# embedding with max pooling
embedding_max = torch.max(hidden_states[0], dim=0)[0]
print(embedding_max.shape) # expect to be 768

OutOfMemoryError: CUDA out of memory. Tried to allocate 9.84 GiB. GPU 3 has a total capacty of 44.34 GiB of which 3.21 GiB is free. Including non-PyTorch memory, this process has 41.12 GiB memory in use. Of the allocated memory 40.33 GiB is allocated by PyTorch, and 353.92 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF